# Validación cruzada de los hiperparámetros (Taiwan)

**Autor:** Herman Echeverría · Grupo Taiwan

**Declaración de uso de IA:** se utilizó Claude (Anthropic) como herramienta de apoyo para comprender lo que pedía cada inciso del enunciado y como asistencia al escribir y depurar el código. El análisis, la verificación de resultados y su interpretación fueron revisados por el autor.

El enunciado pide, desde la Parte 2, ajustar $\lambda$, $k$, la profundidad y `min_samples_leaf` con validación cruzada de 5 pliegues **sobre el entrenamiento** y reportar qué se escogió. Los demás hiperparámetros quedan como en la Parte 1. El criterio es el error cuadrático medio: con $y \in \{0, 1\}$ es el *Brier score* de la probabilidad de mora. Al final se sobrescribe `resultados/hiperparametros.json`, que cargan todos los notebooks siguientes.

In [ ]:
import sys, json
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from datos import cargar_taiwan

X_train, X_test, y_train, y_test, sd = cargar_taiwan("../data/taiwan_credit.csv")
# valores de la Parte 1 (los del enunciado). No se leen del json porque este notebook lo sobrescribe.
hp_inicial = {"lambda": 100, "k": 30, "max_depth": 6, "min_samples_leaf": 5}

## Rejillas

Rejillas amplias, en escala logarítmica cuando el parámetro abarca varios órdenes de magnitud. Si el mejor valor quedara en un borde de la rejilla, habría que ampliarla.

In [ ]:
busquedas = {
    "lambda": (Ridge(), "alpha", np.logspace(-2, 5, 15)),
    "k": (KNeighborsRegressor(), "n_neighbors", [5, 10, 20, 30, 50, 75, 100, 150, 200, 300, 500]),
    "max_depth": (DecisionTreeRegressor(random_state=0), "max_depth", list(range(1, 16))),
    "min_samples_leaf": (RandomForestRegressor(n_estimators=200, max_features=0.33, bootstrap=False,
                                               random_state=0, n_jobs=-1),
                         "min_samples_leaf", [1, 2, 5, 10, 20, 50, 100, 200]),
}

resultados = {}
for nombre, (modelo, param, rejilla) in busquedas.items():
    gs = GridSearchCV(modelo, {param: rejilla}, cv=5, scoring="neg_mean_squared_error").fit(X_train, y_train)
    resultados[nombre] = gs
    print(f"{nombre:>17}: mejor = {gs.best_params_[param]}, MSE de validación = {-gs.best_score_:.5f}")

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (nombre, gs) in zip(ejes, resultados.items()):
    param = busquedas[nombre][1]
    valores = np.array(gs.cv_results_[f"param_{param}"], dtype=float)
    media, de = -gs.cv_results_["mean_test_score"], gs.cv_results_["std_test_score"]
    ax.errorbar(valores, media, yerr=de / np.sqrt(5), marker="o", ms=4, capsize=2)
    ax.axvline(gs.best_params_[param], color="tab:red", ls="--", lw=1)
    if nombre in ("lambda", "k", "min_samples_leaf"):
        ax.set_xscale("log")
    ax.set_title(f"{nombre} (mejor = {gs.best_params_[param]:g})")
    ax.set_xlabel(param)
ejes[0].set_ylabel("MSE de validación (5 pliegues)")
fig.tight_layout()
fig.savefig("../figs/parte2_validacion_cruzada.png", dpi=150)
plt.show()

Las barras son ± un error estándar entre los 5 pliegues. Ningún óptimo quedó en el borde de su rejilla.

- **Ridge:** la curva es plana desde $\lambda = 0.01$ hasta $\lambda \approx 1\,000$ y el mínimo ($\lambda \approx 316$) gana por menos de $10^{-4}$. Con 24 000 filas estandarizadas, los valores propios $d_j^2$ de $X_c^\top X_c$ van de unos 550 a unos 157 000. El factor de encogimiento de cada dirección es $d_j^2/(d_j^2+\lambda)$, así que $\lambda \approx 316$ solo encoge de verdad las 2 o 3 direcciones casi colineales (las de los `BILL_AMT`, que se mueven juntos), justo las que más ruido meten. Ridge casi no se distingue de OLS en estos datos; solo empeora claramente con $\lambda \ge 10^4$.
- **k-NN:** forma de U clara. Con $k$ chico el promedio es muy ruidoso (varianza) y con $k$ grande promedia clientes que ya no se parecen (sesgo). El mínimo está en $k = 50$.
- **Árbol:** el mínimo está en profundidad 4 y desde ahí el error sube rápido: un árbol profundo memoriza el ruido de unas etiquetas 0/1.
- **Bosque:** mejora al subir `min_samples_leaf` de 1 a 20 y se aplana entre 20 y 200. El mínimo está en 50, aunque 20, 50 y 100 quedan dentro de un error estándar entre sí. Hojas más grandes promedian más clientes en cada árbol, lo que ayuda con una respuesta tan ruidosa como la mora.

## Hiperparámetros escogidos

Se sobrescribe `resultados/hiperparametros.json`. Como referencia se reporta también el MSE de prueba con los valores iniciales y con los escogidos (la prueba **no** se usó para escoger nada).